# NOTEBOOK: 03 — LLM Extraction

- **Layer:** Silver
- **Purpose:** For each product in the Silver products table, call the Groq LLM API to extract name, brand and sub-category from the product description.
- **Inputs:** `main.techmart_silver.products`
- **Outputs:** `main.techmart_silver.llm_extracted`
- **Notes:** 
    - Model `llama-3.1-8b-instant` via Groq. 
    - Prompt versioned via Git hash. 
    - All runs traced in MLflow: tokens, latency, success rate, prompt template.
- **Author:** Maira Tavares

# 0. Imports and Configurations

In [0]:
import sys
import json
import time
import mlflow
from pathlib import Path
from pyspark.sql import functions as F

In [0]:
REPO_ROOT = "/Workspace" + str(Path(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()).parent.parent)

from utils.config import (
    # Source and target tables
    SILVER_PRODUCTS,
    LLM_EXTRACTED,
    # LLM model configuration
    LLM_MODEL,
    LLM_API_URL,
    LLM_PROVIDER,
    LLM_MAX_RETRIES,
    LLM_RETRY_DELAY,
    LLM_TIMEOUT,
    # Prompt configuration
    PROMPT_FOLDER,
    PROMPT_EXTRACTION,
    PROMPT_VERSION,
    ALLOWED_SUBCATEGORIES,
    # MLflow
    MLFLOW_EXPERIMENT_NAME,
    # Runtime functions
    get_api_key
)

from utils.states import ProductExtraction
from utils.llm_utils import call_llm, load_prompt, load_prompt_template_raw


In [0]:
PROMPTS_DIR    = Path(REPO_ROOT) / PROMPT_FOLDER   # Path object for / operator
MLFLOW_EXPERIMENT = f"/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}/{MLFLOW_EXPERIMENT_NAME}"

# Runtime values — resolved at execution time
LLM_API_KEY    = get_api_key(dbutils)

In [0]:
LLM_API_KEY

In [0]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 55)
print("NOTEBOOK 03 — LLM EXTRACTION")
print("=" * 55)
print(f"  {'Repo root':<20} : {REPO_ROOT}")
print(f"  {'Prompts dir':<20} : {PROMPTS_DIR}")
print(f"  {'Prompt version':<20} : {PROMPT_VERSION}")
print(f"  {'Model':<20} : {LLM_MODEL}")
print(f"  {'Provider':<20} : {LLM_PROVIDER}")
print(f"  {'Max retries':<20} : {LLM_MAX_RETRIES}")
print(f"  {'Retry delay':<20} : {LLM_RETRY_DELAY}s")
print(f"  {'MLflow experiment':<20} : {MLFLOW_EXPERIMENT}")
print(f"  {'Source table':<20} : {SILVER_PRODUCTS}")
print(f"  {'Target table':<20} : {LLM_EXTRACTED}")
print("=" * 55)
print("✅ Ready to run")

# 1. Functions

In [0]:
# Define extraction function
# This function encapsulates the TechMart-specific extraction logic:
# - Loads system and user prompts from Jinja2 templates
# - Calls the generic LLM utility (provider-agnostic)
# - Validates response structure using Pydantic (ProductExtraction)
# - Returns extracted fields + observability metrics

def extract_product(product_description: str) -> dict:
    """
    Extracts name, brand and sub_category from a raw product description.
    Uses Groq API with automatic retry on rate limit or invalid output.

    Args:
        product_description: Cleaned product description from Silver layer.

    Returns:
        dict with extracted ProductExtraction object + token and latency metrics.

    Raises:
        ValueError: if all retry attempts fail or Pydantic validation fails.
    """
    # Load prompts from versioned Jinja2 templates
    # system prompt sets context and rules
    # user prompt passes the specific product description
    system_prompt = load_prompt(
        PROMPTS_DIR, PROMPT_EXTRACTION, role="system",
        allowed_subcategories=ALLOWED_SUBCATEGORIES
    )
    user_prompt = load_prompt(
        PROMPTS_DIR, PROMPT_EXTRACTION, role="user",
        product_description=product_description
    )

    # Call generic LLM utility — provider-agnostic
    # output_model=ProductExtraction triggers Pydantic validation
    # and automatic retry if the response structure is invalid
    result = call_llm(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        api_key      = LLM_API_KEY,
        api_url      = LLM_API_URL,
        model        = LLM_MODEL,
        max_retries  = LLM_MAX_RETRIES,
        retry_delay  = LLM_RETRY_DELAY,
        timeout      = LLM_TIMEOUT,
        output_model = ProductExtraction
    )

    return {
        "extracted"    : result["validated_output"],
        "input_tokens" : result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "latency"      : result["latency_seconds"]
    }

# 2. Load Data

In [0]:
# Read Silver products
df_products = spark.table(SILVER_PRODUCTS)
print(f"Products to process: {df_products.count()}")
display(df_products.select("product_id", "product_description"))

# 3. Run LLM extraction with MLflow tracing

In [0]:
# Processes all products sequentially with sleep between calls
# to respect Groq free tier TPM rate limits.
# Observability metrics are logged exclusively to MLflow —
# not stored in the Delta table to keep it lean.

mlflow.set_experiment(MLFLOW_EXPERIMENT)

products = df_products.select("product_id", "product_description").collect()
results  = []

with mlflow.start_run(run_name=f"extraction_{PROMPT_VERSION}"):

    # Log prompt template as artifact — captures template structure
    # with Jinja2 placeholders, not rendered values
    mlflow.log_text(
        load_prompt_template_raw(PROMPTS_DIR, PROMPT_EXTRACTION),
        PROMPT_EXTRACTION
    )

    mlflow.log_param("prompt_version", PROMPT_VERSION)
    mlflow.log_param("model",          LLM_MODEL)
    mlflow.log_param("provider",       LLM_PROVIDER)
    mlflow.log_param("total_products", len(products))

    success_count    = 0
    failure_count    = 0
    total_input_tok  = 0
    total_output_tok = 0
    total_latency    = 0.0

    for row in products:
        product_id          = row["product_id"]
        product_description = row["product_description"]

        try:
            result    = extract_product(product_description)
            extracted = result["extracted"]

            total_input_tok  += result["input_tokens"]
            total_output_tok += result["output_tokens"]
            total_latency    += result["latency"]
            success_count    += 1

            results.append({
                "product_id"            : str(product_id),
                "product_description"   : product_description,
                "extracted_name"        : extracted.name,
                "extracted_brand"       : extracted.brand,
                "extracted_sub_category": extracted.sub_category,
                "llm_status"            : "success",
                "prompt_version"        : PROMPT_VERSION
            })

        except Exception as e:
            failure_count += 1
            results.append({
                "product_id"            : str(product_id),
                "product_description"   : product_description,
                "extracted_name"        : None,
                "extracted_brand"       : None,
                "extracted_sub_category": None,
                "llm_status"            : f"failed: {str(e)}",
                "prompt_version"        : PROMPT_VERSION
            })

        # Respect Groq free tier TPM rate limit
        time.sleep(5)

    mlflow.log_metric("success_count",       success_count)
    mlflow.log_metric("failure_count",       failure_count)
    mlflow.log_metric("success_rate",        round(success_count / len(products), 3))
    mlflow.log_metric("total_input_tokens",  total_input_tok)
    mlflow.log_metric("total_output_tokens", total_output_tok)
    mlflow.log_metric("total_tokens",        total_input_tok + total_output_tok)
    mlflow.log_metric("avg_latency_seconds", round(total_latency / len(products), 3))

print(f"✅ Extraction complete")
print(f"   Success : {success_count}")
print(f"   Failed  : {failure_count}")

# 4.  Save and Validate

In [0]:
# Write results to Delta table
# mergeSchema=True allows new fields to be added in future versions
# without requiring a full schema rebuild.
(spark.createDataFrame(results)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(LLM_EXTRACTED))

print(f"✅ Written : {LLM_EXTRACTED}")
print(f"   Rows    : {spark.table(LLM_EXTRACTED).count()}")
display(spark.table(LLM_EXTRACTED))

In [0]:
# Validation summary
print("=" * 55)
print("LLM EXTRACTION — VALIDATION SUMMARY")
print("=" * 55)
print(f"  Total products   : {len(products)}")
print(f"  Success          : {success_count}")
print(f"  Failed           : {failure_count}")
print(f"  Success rate     : {round(success_count / len(products) * 100, 1)}%")
print(f"  Total tokens     : {total_input_tok + total_output_tok}")
print(f"  Avg latency      : {round(total_latency / len(products), 3)}s")
print("=" * 55)
print(f"  MLflow experiment: {MLFLOW_EXPERIMENT}")
print(f"  Prompt version   : {PROMPT_VERSION}")
print("=" * 55)
print("✅ LLM extraction complete — check MLflow for full tracing")